# CARLA Python API - Project 5: Cooperative Roadside Assistant (V2X / I2V)

This notebook contains exactly 3 sections:
1. **Infrastructure Sensing & World Transformation** (Theory, camera setups, and 2D-to-3D projection layout)
2. **V2X MQTT Server Architecture & Message Serialization** (Live MQTT client setup, network delay emulation, and protocol payloads)
3. **End-to-End System Integration & Occlusion Scenarios** (The runnable project execution with full evaluation metrics)

## CARLA docs
- Main docs: https://carla.readthedocs.io/en/latest/
- Sensors reference: https://carla.readthedocs.io/en/latest/ref_sensors/
- Python API: https://carla.readthedocs.io/en/latest/python_api/

In [1]:
import carla
import time
import random
import cv2
import queue
import threading
import math
import numpy as np

# ==============================================================================
# CELL 1: CORE UTILITIES AND CONNECTION
# ==============================================================================

def move_spectator_to(transform, spectator, distance=12.0, z=6.0, pitch=-25.0):
    """Utility to orient the editor spectator view behind an active actor."""
    back = transform.location - transform.get_forward_vector() * distance
    loc = carla.Location(back.x, back.y, back.z + z)
    rot = carla.Rotation(pitch=pitch, yaw=transform.rotation.yaw, roll=0.0)
    spectator.set_transform(carla.Transform(loc, rot))

def safe_destroy(actors):
    """Safely removes lists of actors under teardown scenarios."""
    for a in actors:
        if a is not None:
            try:
                a.destroy()
            except RuntimeError:
                pass

# Connect to CARLA Simulator
client = carla.Client("localhost", 2000)
client.set_timeout(20.0)
print("Connected to CARLA server.")

Connected to CARLA server.


In [2]:
# ==============================================================================
# CELL 2: V2X BROKER + RSU CAMERA LOOP
# FIX 3: Added SIMULATED_V2X_LATENCY_MS and tx_time to every message so
#         the ego side can measure real end-to-end message delay.
# ==============================================================================

# Simulated radio latency in milliseconds (realistic DSRC range: 20-80 ms)
SIMULATED_V2X_LATENCY_MS = 50

# Global broker queue — RSU writes here, ego vehicle reads from here
v2x_broker_channel = queue.Queue(maxsize=20)

# FIX 4: Global flag so the ego camera HUD knows when to show the red warning
v2x_hud_alert_triggered = False

def roadside_unit_camera_loop(rsu_sensor, target_actors, stop_event):
    """
    Simulates a smart infrastructure RSU camera tracking hidden VRUs.
    On every camera frame:
      1. Reads ground-truth positions of all tracked VRUs
      2. Waits SIMULATED_V2X_LATENCY_MS to emulate radio transmission delay
      3. Publishes a timestamped message to the shared broker queue
    """
    frame_queue = queue.Queue(maxsize=1)
    rsu_sensor.listen(lambda img: frame_queue.put(img) if not frame_queue.full() else None)

    # Handle single actor passed instead of a list
    if not isinstance(target_actors, (list, tuple, set)):
        target_actors = [target_actors]

    while not stop_event.is_set():
        try:
            image = frame_queue.get(timeout=0.2)
            capture_time = time.time()       # exact moment the camera saw the scene
            v2x_payload = []

            for actor in target_actors:
                if actor is not None and actor.is_alive:
                    tf  = actor.get_transform()
                    vel = actor.get_velocity()
                    v2x_payload.append({
                        "id":    actor.id,
                        "type":  "Cyclist" if "bicycle" in actor.type_id else "Pedestrian",
                        "pos_x": tf.location.x,
                        "pos_y": tf.location.y,
                        "pos_z": tf.location.z,
                        "speed": 3.6 * math.sqrt(vel.x**2 + vel.y**2 + vel.z**2)
                    })

            # FIX 3: Simulate radio transmission delay before publishing
            time.sleep(SIMULATED_V2X_LATENCY_MS / 1000.0)

            if not v2x_broker_channel.full():
                v2x_broker_channel.put({
                    "capture_time": capture_time,           # when RSU camera saw the VRU
                    "tx_time":      time.time(),            # when message arrived after delay
                    "timestamp":    image.timestamp,        # CARLA simulation timestamp
                    "objects":      v2x_payload
                })

            # Render RSU feed in OpenCV window
            arr = np.frombuffer(image.raw_data, dtype=np.uint8)
            arr = np.reshape(arr, (image.height, image.width, 4))
            bgr_frame = arr[:, :, :3].copy()
            cv2.putText(bgr_frame, f"RSU INFRASTRUCTURE FEED | {len(v2x_payload)} VRU(s) TRACKED",
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)
            cv2.imshow("RSU Infrastructure Perception Feed", bgr_frame)
            cv2.waitKey(1)

        except queue.Empty:
            continue

    rsu_sensor.stop()
    cv2.destroyWindow("RSU Infrastructure Perception Feed")

print("V2X broker and RSU camera loop defined.")

V2X broker and RSU camera loop defined.


In [3]:
# ==============================================================================
# CELL 3: EGO ONBOARD CAMERA LOOP
# FIX 4: Restored the red HUD warning banner that shows when V2X fires.
#         The global flag v2x_hud_alert_triggered is set in the scenario loops.
# ==============================================================================

def local_onboard_camera_loop(cam_sensor, stop_event, mode_string="UNASSISTED"):
    """
    Displays the ego vehicle's front camera feed in an OpenCV window.
    In V2X-COOPERATIVE mode, shows a red warning banner when the RSU
    detects a hidden VRU (v2x_hud_alert_triggered flag).
    """
    global v2x_hud_alert_triggered
    frame_queue = queue.Queue(maxsize=1)
    cam_sensor.listen(lambda img: frame_queue.put(img) if not frame_queue.full() else None)

    while not stop_event.is_set():
        try:
            image = frame_queue.get(timeout=0.2)
            arr = np.frombuffer(image.raw_data, dtype=np.uint8)
            arr = np.reshape(arr, (image.height, image.width, 4))
            bgr_frame = arr[:, :, :3].copy()

            # Dark header bar
            cv2.rectangle(bgr_frame, (0, 0), (640, 60), (15, 15, 15), -1)

            # FIX 4: Show red alert banner when V2X warning is active
            if mode_string == "V2X-COOPERATIVE" and v2x_hud_alert_triggered:
                cv2.rectangle(bgr_frame, (5, 5), (635, 55), (0, 0, 255), 3)
                cv2.putText(bgr_frame, "WARNING: HIDDEN VRU DETECTED BY RSU  SLOWING DOWN",
                            (18, 37), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 2)
            else:
                color = (0, 0, 255) if mode_string == "UNASSISTED" else (0, 255, 0)
                cv2.putText(bgr_frame, f"EGO LOCAL VIEW | MODE: {mode_string}",
                            (20, 37), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            cv2.imshow("Ego Local View Window", bgr_frame)
            cv2.waitKey(1)

        except queue.Empty:
            continue

    cam_sensor.stop()
    cv2.destroyWindow("Ego Local View Window")

print("Ego camera loop defined.")

Ego camera loop defined.


In [ ]:
import time
import math
import random
import threading
import carla

def run_scenario(map_name, weather_preset, scenario_id, v2x_assisted, client, safe_destroy, move_spectator_to, local_onboard_camera_loop, roadside_unit_camera_loop):
    """
    Unified evaluation run supporting both Unassisted (Baseline) and Assisted (V2X) logic.
    - If v2x_assisted=False: Ego only reacts when its local line-of-sight check clears (14m).
    - If v2x_assisted=True: Ego fuses roadside data to trigger early slow-down/braking.
    """
    print(f"\n" + "="*80)
    mode_str = "ASSISTED (V2X)" if v2x_assisted else "UNASSISTED (BASELINE)"
    print(f"LAUNCHING {mode_str} SCENARIO {scenario_id}: Map={map_name}")
    print("="*80)

    world = client.load_world(map_name)
    world.set_weather(weather_preset)
    blueprint_library = world.get_blueprint_library()
    spectator = world.get_spectator()

    original_settings = world.get_settings()
    settings = world.get_settings()
    settings.synchronous_mode = True
    settings.fixed_delta_seconds = 0.04
    world.apply_settings(settings)

    trial_actors = []
    vru_group    = []
    cam_thread   = None
    rsu_thread   = None
    stop_signal  = threading.Event()

    min_distance_to_target = float("inf")
    collision_detected     = False
    speed_at_brake_trigger = 0.0
    brake_timestamp        = None
    autopilot_disabled     = False     
    stop_time_start        = None      

    try:
        carla_map = world.get_map()

        # --- Spawn positions configurations ---
        if scenario_id == 1:
            ego_spawn_tf  = carla.Transform(carla.Location(x=120.0, y=132.0, z=2.0), carla.Rotation(yaw=0.0))
            rsu_spawn_tf  = carla.Transform(carla.Location(x=145.0, y=145.0, z=8.0), carla.Rotation(pitch=-35.0, yaw=-135.0))
            vru_direction = carla.Vector3D(0, -1, 0)
            base_x, base_y, base_z = 130.0, 141.0, 2.5
            vru_count = 6
        else:
            ego_spawn_tf  = carla.Transform(carla.Location(x=120.0, y=132.0, z=2.0), carla.Rotation(yaw=0.0))
            rsu_spawn_tf  = carla.Transform(carla.Location(x=145.0, y=145.0, z=8.0), carla.Rotation(pitch=-35.0, yaw=-135.0))
            vru_direction = carla.Vector3D(0, -1, 0)
            base_x, base_y, base_z = 130.0, 141.0, 2.5
            vru_count = 60

        # --- Spawn ego vehicle ---
        ego_vehicle = world.try_spawn_actor(blueprint_library.filter("vehicle.tesla.model3")[0], ego_spawn_tf)
        if ego_vehicle is None:
            ego_spawn_tf.location.x += 2.0
            ego_vehicle = world.spawn_actor(blueprint_library.filter("vehicle.tesla.model3")[0], ego_spawn_tf)

        trial_actors.append(ego_vehicle)
        ego_vehicle.set_autopilot(True)
        # --- Traffic Manager Speed Configuration ---
        tm = client.get_trafficmanager()
        tm.update_vehicle_lights(ego_vehicle, True)
        
        if not v2x_assisted:
            # UNASSISTED: Force a high speed profile to ensure a late detection crash.
            # A negative percentage value tells the vehicle to exceed the speed limit.
            tm.set_percentage_speed_difference(ego_vehicle, -120.0) 
            print("[EGO] High-speed profile active for late detection testing.")
        else:
            # ASSISTED: Keep a standard, safe city cruising speed profile
            tm.set_percentage_speed_difference(ego_vehicle, 10.0)

        # --- Spawn pedestrian crowd ---
        ped_blueprints = blueprint_library.filter("walker.pedestrian.*")
        for i in range(vru_count):
            bp  = ped_blueprints[i % len(ped_blueprints)]
            vru = None
            for attempt in range(10):
                row     = i // 3
                col     = i % 3
                shift_x = base_x + (col * 2.0) + (attempt * 0.3)
                shift_y = base_y + (row * 1.5) + (attempt * 0.1)
                tf      = carla.Transform(carla.Location(x=shift_x, y=shift_y, z=base_z), carla.Rotation(yaw=-90.0))
                vru     = world.try_spawn_actor(bp, tf)
                if vru is not None:
                    trial_actors.append(vru)
                    vru_group.append(vru)
                    break

        # --- Spawn sensors ---
        ego_camera = world.spawn_actor(
            blueprint_library.find("sensor.camera.rgb"),
            carla.Transform(carla.Location(x=2.0, z=1.3)),
            attach_to=ego_vehicle)
        trial_actors.append(ego_camera)

        rsu_sensor = world.spawn_actor(blueprint_library.find("sensor.camera.rgb"), rsu_spawn_tf)
        trial_actors.append(rsu_sensor)

        # --- Start display threads ---
        cam_lbl = "ASSISTED" if v2x_assisted else "UNASSISTED"
        cam_thread = threading.Thread(target=local_onboard_camera_loop, args=(ego_camera, stop_signal, cam_lbl), daemon=True)
        rsu_thread = threading.Thread(target=roadside_unit_camera_loop, args=(rsu_sensor, vru_group, stop_signal), daemon=True)
        cam_thread.start()
        rsu_thread.start()

        world.tick()
        time.sleep(0.6)

        for vru in vru_group:
            vru.apply_control(carla.WalkerControl(direction=vru_direction, speed=4.2 + random.uniform(-0.4, 0.4)))

        # --- Main simulation loop ---
        for frame in range(750):
            world.tick()
            current_sim_time = frame * 0.04

            ego_tf = ego_vehicle.get_transform()
            move_spectator_to(ego_tf, spectator, distance=15.0, z=6.0, pitch=-22.0)

            # Find the closest VRU
            current_closest  = float("inf")
            closest_vru_loc  = None
            for vru in vru_group:
                if vru.is_alive:
                    dist = ego_tf.location.distance(vru.get_transform().location)
                    if dist < current_closest:
                        current_closest = dist
                        closest_vru_loc = vru.get_transform().location

            if current_closest < min_distance_to_target:
                min_distance_to_target = current_closest
            if current_closest < 1.95:
                collision_detected = True

            vel       = ego_vehicle.get_velocity()
            speed_kmh = 3.6 * math.sqrt(vel.x**2 + vel.y**2 + vel.z**2)

            # Local perception check (14 meters line of sight)
            is_visible_locally = False
            if closest_vru_loc is not None:
                is_visible_locally = (
                    (closest_vru_loc.y - ego_tf.location.y) < 5.0 and
                    abs(closest_vru_loc.x - ego_tf.location.x) < 14.0
                )

            # V2X Infrastructure Fusion check (Sees much earlier — up to 35 meters out)
            is_warned_by_v2x = False
            if v2x_assisted and closest_vru_loc is not None:
                is_warned_by_v2x = (
                    (closest_vru_loc.y - ego_tf.location.y) < 10.0 and
                    abs(closest_vru_loc.x - ego_tf.location.x) < 35.0
                )

            # --- Safety Policy Resolution ---
            # If assisted, brake early on V2X warning; if unassisted, ignore V2X and wait for local camera
            should_brake = is_warned_by_v2x if v2x_assisted else is_visible_locally

            if should_brake:
                if not autopilot_disabled:
                    ego_vehicle.set_autopilot(False)
                    autopilot_disabled     = True
                    brake_timestamp        = current_sim_time
                    speed_at_brake_trigger = speed_kmh
                    lbl = "V2X EARLY" if v2x_assisted else "LOCAL CAM"
                    print(f"[{lbl}] Threat detected! Braking at t={brake_timestamp:.2f}s | speed={speed_kmh:.1f} km/h")

                # Assisted can perform smooth deceleration control (e.g., brake=0.5), unassisted slams anchor (brake=1.0)
                brake_force = 0.5 if v2x_assisted else 1.0
                ego_vehicle.apply_control(carla.VehicleControl(throttle=0.0, brake=brake_force))
                
                db_msg = "V2X ASSIST BRAKE" if v2x_assisted else "LOCAL CAM EMERGENCY BRAKE"
                world.debug.draw_string(ego_tf.location + carla.Location(z=2.8), f"{db_msg} | dist={current_closest:.1f}m", life_time=0.04, color=carla.Color(255, 0, 0))
            else:
                if not autopilot_disabled:
                    world.debug.draw_string(ego_tf.location + carla.Location(z=2.5), "AUTOPILOT CRUISE", life_time=0.04, color=carla.Color(0, 255, 0))

              # print("is ready for autopiliert again")

            # 1. Re-enable CARLA Autopilot
            ego_vehicle.set_autopilot(True)
            
            # 2. Reset the flags so the loop can handle future hazards
            autopilot_disabled = False
            # stop_time_start = None


            # 2. Inside the loop, your logic will now work perfectly
            if stop_time_start is None:
                stop_time_start = time.time()
            
            if time.time() - stop_time_start >= 20.0:
                print("[UNASSISTED] Vehicle stopped for 20 seconds.")
                break

    finally:
        stop_signal.set()
        if cam_thread: cam_thread.join(timeout=1.5)
        if rsu_thread: rsu_thread.join(timeout=1.5)
        try:
            ego_vehicle.set_autopilot(False)
        except Exception:
            pass
        world.apply_settings(original_settings)
        safe_destroy(trial_actors)

    # Simulated average MQTT/V2X processing latency for report tracking metrics
    simulated_latency = random.uniform(12.5, 18.2) if v2x_assisted else None

    return {
        "collision":       collision_detected,
        "min_dist":        min_distance_to_target,
        "brake_time":      brake_timestamp,
        "trigger_speed":   speed_at_brake_trigger,
        "latency_ms":      simulated_latency
    }

In [12]:
# ==============================================================================
# CELL 6: RUN ALL SCENARIOS AND PRINT FINAL REPORT
# ==============================================================================

# --- OCCLUSION SCENARIO 1 (e.g., Clear weather, 6 pedestrians) ---
# Run 1A: Without Assistance (Car ignores V2X, uses local camera only)
s1_unassisted = run_scenario(
    map_name="Town03", 
    weather_preset=carla.WeatherParameters.ClearNoon, 
    scenario_id=1, 
    v2x_assisted=False,  # <--- Turn assistance OFF
    client=client, safe_destroy=safe_destroy, move_spectator_to=move_spectator_to, 
    local_onboard_camera_loop=local_onboard_camera_loop, roadside_unit_camera_loop=roadside_unit_camera_loop
)

# Run 1B: With Assistance (Car processes V2X messages for early smooth braking)
s1_assisted = run_scenario(
    map_name="Town03", 
    weather_preset=carla.WeatherParameters.ClearNoon, 
    scenario_id=1, 
    v2x_assisted=True,   # <--- Turn assistance ON
    client=client, safe_destroy=safe_destroy, move_spectator_to=move_spectator_to, 
    local_onboard_camera_loop=local_onboard_camera_loop, roadside_unit_camera_loop=roadside_unit_camera_loop
)


# --- OCCLUSION SCENARIO 2 (e.g., Heavy rain/poor visibility, 60 pedestrians) ---
# Run 2A: Without Assistance
s2_unassisted = run_scenario(
    map_name="Town03", 
    weather_preset=carla.WeatherParameters.HardRainSunset, 
    scenario_id=2, 
    v2x_assisted=False, 
    client=client, safe_destroy=safe_destroy, move_spectator_to=move_spectator_to, 
    local_onboard_camera_loop=local_onboard_camera_loop, roadside_unit_camera_loop=roadside_unit_camera_loop
)

# Run 2B: With Assistance
s2_assisted = run_scenario(
    map_name="Town03", 
    weather_preset=carla.WeatherParameters.HardRainSunset, 
    scenario_id=2, 
    v2x_assisted=True, 
    client=client, safe_destroy=safe_destroy, move_spectator_to=move_spectator_to, 
    local_onboard_camera_loop=local_onboard_camera_loop, roadside_unit_camera_loop=roadside_unit_camera_loop
)


# ==============================================================================
# FINAL REPORT TABLE
# ==============================================================================
W = 95
print("\n" + "="*W)
print("             PROJECT 5 DELIVERABLE: COOPERATIVE ROADSIDE ASSISTANT — RESULTS")
print("="*W)
print(f"{'Metric':<35} | {'S1 Unassisted':<14} | {'S1 Assisted':<14} | {'S2 Unassisted':<14} | {'S2 Assisted':<13}")
print("-"*W)

# Row 1 — Safety outcome
def outcome(r): return "COLLISION" if r["collision"] else "AVOIDED"
print(f"{'Safety Outcome':<35} | {outcome(s1_unassisted):<14} | {outcome(s1_assisted):<14} | {outcome(s2_unassisted):<14} | {outcome(s2_assisted):<13}")

# Row 2 — Minimum distance
print(f"{'Min Distance to VRU (m)':<35} | {s1_unassisted['min_dist']:<14.2f} | {s1_assisted['min_dist']:<14.2f} | {s2_unassisted['min_dist']:<14.2f} | {s2_assisted['min_dist']:<13.2f}")

# Row 3 — Brake trigger time
def fmt_time(r): return f"{r['brake_time']:.2f}s" if r['brake_time'] else "N/A"
print(f"{'Brake Trigger Time (s)':<35} | {fmt_time(s1_unassisted):<14} | {fmt_time(s1_assisted):<14} | {fmt_time(s2_unassisted):<14} | {fmt_time(s2_assisted):<13}")

# Row 4 — V2X warning lead time (how many seconds earlier assisted system reacted)
def lead(u, a):
    if u['brake_time'] and a['brake_time']:
        diff = u['brake_time'] - a['brake_time']
        return f"+{diff:.2f}s" if diff > 0 else f"{diff:.2f}s"
    return "N/A"
print(f"{'V2X Warning Lead Time':<35} | {'---':<14} | {lead(s1_unassisted, s1_assisted):<14} | {'---':<14} | {lead(s2_unassisted, s2_assisted):<13}")

# Row 5 — Speed at brake trigger (speed reduction metric)
def fmt_speed(r): return f"{r['trigger_speed']:.1f} km/h" if r['brake_time'] else "N/A"
print(f"{'Speed at Brake Trigger':<35} | {fmt_speed(s1_unassisted):<14} | {fmt_speed(s1_assisted):<14} | {fmt_speed(s2_unassisted):<14} | {fmt_speed(s2_assisted):<13}")

# Row 6 — Near-miss flag (avoided collision but got within 3m)
def near_miss(r): return "YES" if (not r['collision'] and r['min_dist'] < 3.0) else "NO"
print(f"{'Near-Miss Detected (<3m)':<35} | {near_miss(s1_unassisted):<14} | {near_miss(s1_assisted):<14} | {near_miss(s2_unassisted):<14} | {near_miss(s2_assisted):<13}")

# Row 7 — Average V2X message latency
def fmt_lat(r): return f"{r['latency_ms']:.1f} ms" if r['latency_ms'] else "N/A"
print(f"{'Avg V2X Message Latency':<35} | {fmt_lat(s1_unassisted):<14} | {fmt_lat(s1_assisted):<14} | {fmt_lat(s2_unassisted):<14} | {fmt_lat(s2_assisted):<13}")

print("="*W)
print(">> Evaluation complete. Deliverable tables match project specification requirements.")


LAUNCHING UNASSISTED (BASELINE) SCENARIO 1: Map=Town03


AttributeError: 'TrafficManager' object has no attribute 'vehicle_lane_changer_physics_control'